# Connecting Budget Bill to Fiscal Data
These codes in this file aim to connect Budget Bill to Fiscal Data in fiscal year 2023 (July 2023 - June 2024)

## 1. Implement by using FY2023P01 only

In [1]:
import pandas as pd

In [2]:
# Designate two target csv files
Bill = "2023_Budget_Final_Amounts.csv"
Fiscal = "Vendor_FY23P01.csv"

In [3]:
# Convert csv files into dataframes
bill_df = pd.read_csv(Bill)
bill_df.head()

,item_number,program_code,object_code,fund_code,revised_amount
0,0110-001-0001,960.0,101001.0,1.0,-6751000
1,0110-001-0001,960.0,317292.0,1.0,-1712000
2,0110-001-0001,960.0,317295.0,1.0,-11000
3,0110-001-0001,960.0,500004.0,1.0,-168851000
4,0110-001-0001,960.0,EMPTY,1.0,177325000


In [4]:
fiscal_df = pd.read_csv(Fiscal)
fiscal_df.head()

C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\1187446066.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  fiscal_df = pd.read_csv(Fiscal)


,business_unit,agency_name,department_name,document_id,related_document,accounting_date,fiscal_year_begin,accounting_period,VENDOR_NAME,account,...,fund_description,program_code,program_description,sub_program_description,budget_reference,budget_reference_category,budget_reference_sub_category,budget_reference_description,year_of_enactment,monetary_amount
0,974,"Legislative, Judicial, & Exec",Pollution Control Fin Auth,0974.00001628.0.00001.0006,NaN,2023/7/14,2023,1,CONFIDENTIAL,5320440,...,Pollution Control Financing Au,9990000000,Unscheduled Items of Approp,Unscheduled Items of Approp,501,State Operations,Non-Budget Act,Non-BA State Operations-Sup501,1979,44.57
1,954,"Legislative, Judicial, & Exec",Scholarshare Investment Board,0954.00002207.0.00001.0002,NaN,2023/7/17,2023,1,CONFIDENTIAL,5320810,...,Scholarshare Administrative Fu,780000000,Golden St Scholarshare Trust,Golden St Scholarshare Trust,1,State Operations,Budget Act,BA State Operations-Support001,2022,72.87
2,3340,Natural Resources,CA Conservation Corps,3340.00087879.0.00001.0001,NaN,2023/7/14,2023,1,DEPT OF GENERAL SERVICES,5360350,...,Public Bldg Construction Fund,2365000000,Capital Outlay,Capital Outlay,301,Capital Outlay,Budget Act,BA Capital Outlay Project 301,2020,174437.50
3,7870,Government Operations,Victim Compensation Board CA,7870.00018620.0.00001.0001,NaN,2023/7/14,2023,1,DEPT OF STATE HOSPITALS,5340220,...,Forced or Involuntary Steril,6380000000,Victim Compensation,Victim Compensation,501,State Operations,Non-Budget Act,Non-BA State Operations-Sup501,2021,973.50
4,3600,Natural Resources,Department of Fish & Wildlife,3600.00296274.0.00002.0001,NaN,2023/7/5,2023,1,DEPT OF WATER RESOURCES,5340330,...,Salton Sea Restoration Fund,2625000000,Capital Outlay,Capital Outlay,301,Capital Outlay,Budget Act,BA Capital Outlay Project 301,2019,1388933.76


In [5]:
# Extract a few columns from each dataframe to create two new dataframes
bill_df_new = bill_df[["item_number", "program_code", "object_code", "revised_amount"]]
bill_df_new.head()

,item_number,program_code,object_code,revised_amount
0,0110-001-0001,960.0,101001.0,-6751000
1,0110-001-0001,960.0,317292.0,-1712000
2,0110-001-0001,960.0,317295.0,-11000
3,0110-001-0001,960.0,500004.0,-168851000
4,0110-001-0001,960.0,EMPTY,177325000


In [6]:
fiscal_df_new = fiscal_df[["business_unit", "budget_reference", "fund_code", "program_code", "monetary_amount", "year_of_enactment"]]
fiscal_df_new.head()

,business_unit,budget_reference,fund_code,program_code,monetary_amount,year_of_enactment
0,974,501,930,9990000000,44.57,1979
1,954,1,564,780000000,72.87,2022
2,3340,301,660,2365000000,174437.50,2020
3,7870,501,3383,6380000000,973.50,2021
4,3600,301,8018,2625000000,1388933.76,2019


In [7]:
# Split three sections in "item_number" column in bill_df_new into three new columns: "item_number_1", "item_number_2", and "item_number_3"
bill_df_new[["business_unit", "budget_reference", "fund_code"]] = bill_df_new["item_number"].str.split("-", expand=True)
bill_df_new = bill_df_new.drop(columns=["item_number"])
bill_df_new.head()

,program_code,object_code,revised_amount,business_unit,budget_reference,fund_code
0,960.0,101001.0,-6751000,0110,001,0001
1,960.0,317292.0,-1712000,0110,001,0001
2,960.0,317295.0,-11000,0110,001,0001
3,960.0,500004.0,-168851000,0110,001,0001
4,960.0,EMPTY,177325000,0110,001,0001


In [8]:
# check data types
print(bill_df_new.dtypes)
# convert all columns to interger type if the data is not NaN
bill_df_new = bill_df_new.apply(pd.to_numeric, errors='coerce')
bill_df_new.head()

program_code        object
object_code         object
revised_amount       int64
business_unit       object
budget_reference    object
fund_code           object
dtype: object


,program_code,object_code,revised_amount,business_unit,budget_reference,fund_code
0,960.0,101001.0,-6751000,110,1,1
1,960.0,317292.0,-1712000,110,1,1
2,960.0,317295.0,-11000,110,1,1
3,960.0,500004.0,-168851000,110,1,1
4,960.0,NaN,177325000,110,1,1


In [9]:
fiscal_df_new['program_code'] = fiscal_df_new['program_code'].astype(str)

# Create two new columns "prog_4" and "obj_7" in fiscal_df_new by extracting the first 4 and 7 characters from the "program_code" column, respectively
fiscal_df_new['prog_4'] = fiscal_df_new['program_code'].str[:4]
fiscal_df_new['obj_7']  = fiscal_df_new['program_code'].str[:7]

C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\3312172717.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fiscal_df_new['program_code'] = fiscal_df_new['program_code'].astype(str)
C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\3312172717.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fiscal_df_new['prog_4'] = fiscal_df_new['program_code'].str[:4]
C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\3312172717.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [10]:
# Just use "year_of_enactment" is 2023 in fiscal_df_new
fiscal_df_2023 = fiscal_df_new[fiscal_df_new["year_of_enactment"] == 2023]
fiscal_df_2023.head()

,business_unit,budget_reference,fund_code,program_code,monetary_amount,year_of_enactment,prog_4,obj_7
11,2670,1,290,2030010000,23206.21,2023,2030,2030010
96,7350,501,223,6120000000,1960000.00,2023,6120,6120000
148,7350,501,223,6120000000,2440000.00,2023,6120,6120000
265,3780,1,1,2830000000,10.35,2023,2830,2830000
272,2670,1,290,2030019000,2204.78,2023,2030,2030019


In [11]:
# --- 1. Rename columns and create join key ---
bill_df_new = bill_df_new.rename(columns={'program_code': 'bill_project_code'})

def get_bill_join_key(row):
    obj = str(row.get('object_code', '')).replace('.0', '').strip()
    prog = str(row.get('bill_project_code', '')).replace('.0', '').strip()
    
    if obj in ['0', 'nan', 'EMPTY', '']:
        return prog
    return obj

bill_df_new['join_code'] = bill_df_new.apply(get_bill_join_key, axis=1)

# --- 2. Clean and prepare fiscal_df_2023 for merging ---
for col in ["business_unit", "budget_reference", "fund_code"]:
    bill_df_new[col] = bill_df_new[col].fillna(0).astype(int).astype(str)
    fiscal_df_2023[col] = fiscal_df_2023[col].fillna(0).astype(int).astype(str)

# Create "prog_clean" in fiscal_df_2023 by removing ".0" and stripping whitespace, then create "prog_4" and "obj_7" from "prog_clean"
fiscal_df_2023['prog_clean'] = fiscal_df_2023['program_code'].astype(str).str.replace('.0', '').str.strip()
fiscal_df_2023['prog_4'] = fiscal_df_2023['prog_clean'].str[:4]
fiscal_df_2023['obj_7'] = fiscal_df_2023['prog_clean'].str[:7]

# --- 3. Merge ---
merged_df = pd.merge(
    bill_df_new, 
    fiscal_df_2023, 
    left_on=["business_unit", "budget_reference", "fund_code", "join_code"],
    right_on=["business_unit", "budget_reference", "fund_code", "obj_7"],
    how="left"
)

unmatched = merged_df[merged_df['monetary_amount'].isna()].copy()
matched = merged_df[merged_df['monetary_amount'].notna()].copy()

if not unmatched.empty:
    unmatched_clean = unmatched[bill_df_new.columns]
    unmatched_fixed = pd.merge(
        unmatched_clean,
        fiscal_df_2023,
        left_on=["business_unit", "budget_reference", "fund_code", "join_code"],
        right_on=["business_unit", "budget_reference", "fund_code", "prog_4"],
        how="left"
    )
    merged_df = pd.concat([matched, unmatched_fixed], ignore_index=True)

print(f"The total number of rows in the merged DataFrame is: {len(merged_df)}")
merged_df.head()

The total number of rows in the merged DataFrame is: 11527


C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\2813747369.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fiscal_df_2023[col] = fiscal_df_2023[col].fillna(0).astype(int).astype(str)
C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\2813747369.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fiscal_df_2023[col] = fiscal_df_2023[col].fillna(0).astype(int).astype(str)
C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\2813747369.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy o

,bill_project_code,object_code,revised_amount,business_unit,budget_reference,fund_code,join_code,program_code,monetary_amount,year_of_enactment,prog_4,obj_7,prog_clean
0,NaN,1200010.0,3365000,1111,1,264,1200010,1200010000,225.00,2023.0,1200,1200010,1200010000
1,NaN,1130010.0,78520000,1111,1,735,1130010,1130010000,3965.55,2023.0,1130,1130010,1130010000
2,NaN,1130010.0,78520000,1111,1,735,1130010,1130010000,52500.00,2023.0,1130,1130010,1130010000
3,NaN,1130010.0,78520000,1111,1,735,1130010,1130010000,1356.08,2023.0,1130,1130010,1130010000
4,NaN,1130010.0,78520000,1111,1,735,1130010,1130010000,5556.60,2023.0,1130,1130010,1130010000


In [12]:
# --- 1. Rename columns and create join key ---
if 'year_of_enactment_x' in merged_df.columns:
    merged_df['Budget_Enactment_Year'] = merged_df['year_of_enactment_x']
else:
    merged_df['Budget_Enactment_Year'] = 2023 

merged_df = merged_df.rename(columns={
    'revised_amount': 'Budget_Amount',
    'monetary_amount': 'Actual_Spending',
    'accounting_date': 'Payment_Date',
    'program_code_y': 'Actual_Program_Code' 
})

# --- 2. Reorder columns ---
ordered_columns = [
    'Budget_Enactment_Year', 
    'business_unit',         
    'budget_reference',      
    'fund_code',             
    'join_code',             
    'Budget_Amount',         
    'Actual_Spending',       
    'Payment_Date',         
    'VENDOR_NAME',           
    'Actual_Program_Code',   
    'program_description'    
]

existing_cols = [c for c in ordered_columns if c in merged_df.columns]
remaining_cols = [c for c in merged_df.columns if c not in existing_cols]

merged_df = merged_df[existing_cols + remaining_cols]

# --- 3. Sort by Budget_Enactment_Year, business_unit, and join_code ---
merged_df = merged_df.sort_values(['Budget_Enactment_Year', 'business_unit', 'join_code'])

print(f"The total number of columns in the merged DataFrame is: {len(merged_df.columns)}")
merged_df.head()

The total number of columns in the merged DataFrame is: 14


,Budget_Enactment_Year,business_unit,budget_reference,fund_code,join_code,Budget_Amount,Actual_Spending,bill_project_code,object_code,program_code,year_of_enactment,prog_4,obj_7,prog_clean
6369,2023,1045,1,3288,1045,3264000,NaN,1045.0,NaN,NaN,NaN,NaN,NaN,NaN
5976,2023,110,1,1,101001,-6751000,NaN,960.0,101001.0,NaN,NaN,NaN,NaN,NaN
5977,2023,110,1,1,317292,-1712000,NaN,960.0,317292.0,NaN,NaN,NaN,NaN,NaN
5978,2023,110,1,1,317295,-11000,NaN,960.0,317295.0,NaN,NaN,NaN,NaN,NaN
5979,2023,110,1,1,500004,-168851000,NaN,960.0,500004.0,NaN,NaN,NaN,NaN,NaN


In [13]:
# 1. Extract necessary columns for crosswalk export
actual_prog_col = 'Actual_Program_Code' if 'Actual_Program_Code' in merged_df.columns else 'program_code'

crosswalk_export = merged_df[[
    'Budget_Enactment_Year',
    'business_unit',
    'budget_reference',
    'fund_code',
    'join_code',       
    actual_prog_col
]].drop_duplicates().copy()

# 2. Create "Mapping Level" column based on the length of "join_code"
def get_mapping_level(code):
    code_str = str(code).replace('.0', '')
    length = len(code_str)
    if length <= 4:
        return f"{length}-digit (Project) Match"
    else:
        return f"{length}-digit (Object) Match"

crosswalk_export['Mapping_Level'] = crosswalk_export['join_code'].apply(get_mapping_level)

# 3. Change column names to match the required output format
crosswalk_export = crosswalk_export.rename(columns={
    'Budget_Enactment_Year': 'year_of_enactment',
    'join_code': 'Budget_Target_Code',
    actual_prog_col: 'Actual_Full_Code_9digit'
})

# 4. Create "Budget_ID" by concatenating the relevant columns with hyphens
crosswalk_export['Budget_ID'] = (
    crosswalk_export['year_of_enactment'].astype(str) + "-" + 
    crosswalk_export['business_unit'].astype(str) + "-" + 
    crosswalk_export['budget_reference'].astype(str) + "-" + 
    crosswalk_export['fund_code'].astype(str) + "-" + 
    crosswalk_export['Budget_Target_Code'].astype(str)
)

# 5. Output to CSV
crosswalk_export.to_csv("Crosswalk_only_FY23P01.csv", index=False, encoding='utf-8-sig')

# Check the final output
print(f"Final output rows: {len(crosswalk_export)}")
print("Budget_ID samples:")
print(crosswalk_export['Budget_ID'].head())

Final output rows: 2617
Budget_ID samples:
6369    2023-1045-1-3288-1045
5976      2023-110-1-1-101001
5977      2023-110-1-1-317292
5978      2023-110-1-1-317295
5979      2023-110-1-1-500004
Name: Budget_ID, dtype: object


## Connect two datasets by Budget_ID

In [14]:
crosswalk_export.head()

,year_of_enactment,business_unit,budget_reference,fund_code,Budget_Target_Code,Actual_Full_Code_9digit,Mapping_Level,Budget_ID
6369,2023,1045,1,3288,1045,NaN,4-digit (Project) Match,2023-1045-1-3288-1045
5976,2023,110,1,1,101001,NaN,6-digit (Object) Match,2023-110-1-1-101001
5977,2023,110,1,1,317292,NaN,6-digit (Object) Match,2023-110-1-1-317292
5978,2023,110,1,1,317295,NaN,6-digit (Object) Match,2023-110-1-1-317295
5979,2023,110,1,1,500004,NaN,6-digit (Object) Match,2023-110-1-1-500004


In [15]:
Fiscal = "Vendor_FY23P01.csv"
fiscal_df_to_connect = pd.read_csv(Fiscal)
fiscal_df_to_connect.head()

C:\Users\mshun\AppData\Local\Temp\ipykernel_28164\4154011424.py:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  fiscal_df_to_connect = pd.read_csv(Fiscal)


,business_unit,agency_name,department_name,document_id,related_document,accounting_date,fiscal_year_begin,accounting_period,VENDOR_NAME,account,...,fund_description,program_code,program_description,sub_program_description,budget_reference,budget_reference_category,budget_reference_sub_category,budget_reference_description,year_of_enactment,monetary_amount
0,974,"Legislative, Judicial, & Exec",Pollution Control Fin Auth,0974.00001628.0.00001.0006,NaN,2023/7/14,2023,1,CONFIDENTIAL,5320440,...,Pollution Control Financing Au,9990000000,Unscheduled Items of Approp,Unscheduled Items of Approp,501,State Operations,Non-Budget Act,Non-BA State Operations-Sup501,1979,44.57
1,954,"Legislative, Judicial, & Exec",Scholarshare Investment Board,0954.00002207.0.00001.0002,NaN,2023/7/17,2023,1,CONFIDENTIAL,5320810,...,Scholarshare Administrative Fu,780000000,Golden St Scholarshare Trust,Golden St Scholarshare Trust,1,State Operations,Budget Act,BA State Operations-Support001,2022,72.87
2,3340,Natural Resources,CA Conservation Corps,3340.00087879.0.00001.0001,NaN,2023/7/14,2023,1,DEPT OF GENERAL SERVICES,5360350,...,Public Bldg Construction Fund,2365000000,Capital Outlay,Capital Outlay,301,Capital Outlay,Budget Act,BA Capital Outlay Project 301,2020,174437.50
3,7870,Government Operations,Victim Compensation Board CA,7870.00018620.0.00001.0001,NaN,2023/7/14,2023,1,DEPT OF STATE HOSPITALS,5340220,...,Forced or Involuntary Steril,6380000000,Victim Compensation,Victim Compensation,501,State Operations,Non-Budget Act,Non-BA State Operations-Sup501,2021,973.50
4,3600,Natural Resources,Department of Fish & Wildlife,3600.00296274.0.00002.0001,NaN,2023/7/5,2023,1,DEPT OF WATER RESOURCES,5340330,...,Salton Sea Restoration Fund,2625000000,Capital Outlay,Capital Outlay,301,Capital Outlay,Budget Act,BA Capital Outlay Project 301,2019,1388933.76


In [16]:
bill_df_new.head()

,bill_project_code,object_code,revised_amount,business_unit,budget_reference,fund_code,join_code
0,960.0,101001.0,-6751000,110,1,1,101001
1,960.0,317292.0,-1712000,110,1,1,317292
2,960.0,317295.0,-11000,110,1,1,317295
3,960.0,500004.0,-168851000,110,1,1,500004
4,960.0,NaN,177325000,110,1,1,960


In [17]:
import pandas as pd

# --- 1. Define a function to clean and pad codes ---
def pad_code(val, length):
    if pd.isna(val) or val == '' or str(val).lower() == 'nan':
        return '0'.zfill(length)
    try:
        clean_val = str(int(float(val)))
        return clean_val.zfill(length)
    except:
        return str(val).zfill(length)

# --- 2. Clean and prepare bill_df_new for merging ---
bill_df_new['clean_bu'] = bill_df_new['business_unit'].apply(lambda x: pad_code(x, 4))
bill_df_new['clean_ref'] = bill_df_new['budget_reference'].apply(lambda x: pad_code(x, 3))
bill_df_new['clean_fund'] = bill_df_new['fund_code'].apply(lambda x: pad_code(x, 4))

def get_final_bill_code(row):
    obj = str(row.get('object_code', '')).replace('.0', '').strip()
    proj = str(row.get('bill_project_code', '')).replace('.0', '').strip()
    target = obj if (obj and obj not in ['0', 'nan', 'EMPTY']) else proj
    return target if target else '0'

bill_df_new['Budget_ID'] = (
    "2023-" + 
    bill_df_new['clean_bu'] + "-" + 
    bill_df_new['clean_ref'] + "-" + 
    bill_df_new['clean_fund'] + "-" + 
    bill_df_new.apply(get_final_bill_code, axis=1)
)

# --- 3. Create "Budget_ID" for fiscal_df_to_connect ---
fiscal_2023 = fiscal_df_to_connect[fiscal_df_to_connect['year_of_enactment'] == 2023].copy()

fiscal_2023['clean_bu'] = fiscal_2023['business_unit'].apply(lambda x: pad_code(x, 4))
fiscal_2023['clean_ref'] = fiscal_2023['budget_reference'].apply(lambda x: pad_code(x, 3))
fiscal_2023['clean_fund'] = fiscal_2023['fund_code'].apply(lambda x: pad_code(x, 4))

fiscal_2023['prog_str'] = fiscal_2023['program_code'].astype(str).str.replace('.0', '').str.strip()
fiscal_2023['obj_7'] = fiscal_2023['prog_str'].str[:7]
fiscal_2023['proj_4'] = fiscal_2023['prog_str'].str[:4]

fiscal_2023['Budget_ID_7'] = "2023-" + fiscal_2023['clean_bu'] + "-" + fiscal_2023['clean_ref'] + "-" + fiscal_2023['clean_fund'] + "-" + fiscal_2023['obj_7']
fiscal_2023['Budget_ID_4'] = "2023-" + fiscal_2023['clean_bu'] + "-" + fiscal_2023['clean_ref'] + "-" + fiscal_2023['clean_fund'] + "-" + fiscal_2023['proj_4']

# --- 4. Check the results ---
print("--- Budget_ID samples (e.g., 2023-0110-001-0001-101001) ---")
print(bill_df_new['Budget_ID'].head().tolist())

print("\n--- Fiscal ID samples (7-digit version) ---")
print(fiscal_2023['Budget_ID_7'].head().tolist())

--- Budget_ID samples (e.g., 2023-0110-001-0001-101001) ---
['2023-0110-001-0001-101001', '2023-0110-001-0001-317292', '2023-0110-001-0001-317295', '2023-0110-001-0001-500004', '2023-0110-001-0001-960']

--- Fiscal ID samples (7-digit version) ---
['2023-2670-001-0290-2030010', '2023-7350-501-0223-6120000', '2023-7350-501-0223-6120000', '2023-3780-001-0001-2830000', '2023-2670-001-0290-2030019']


In [18]:
print("--- Budget Master (bill_df_new) Columns ---")
print(bill_df_new.columns.tolist())

print("\n--- Expenditure Details (fiscal_df_to_connect) Columns ---")
print(fiscal_df_to_connect.columns.tolist())

--- Budget Master (bill_df_new) Columns ---
['bill_project_code', 'object_code', 'revised_amount', 'business_unit', 'budget_reference', 'fund_code', 'join_code', 'clean_bu', 'clean_ref', 'clean_fund', 'Budget_ID']

--- Expenditure Details (fiscal_df_to_connect) Columns ---
['business_unit', 'agency_name', 'department_name', 'document_id', 'related_document', 'accounting_date', 'fiscal_year_begin', 'accounting_period', 'VENDOR_NAME', 'account', 'account_type', 'account_category', 'account_sub_category', 'account_description', 'fund_code', 'fund_group', 'fund_description', 'program_code', 'program_description', 'sub_program_description', 'budget_reference', 'budget_reference_category', 'budget_reference_sub_category', 'budget_reference_description', 'year_of_enactment', 'monetary_amount']


In [19]:
# --- 1. Merge on Budget_ID_7 (Object Code level) ---
merged_step1 = pd.merge(
    bill_df_new, 
    fiscal_2023, 
    left_on='Budget_ID', 
    right_on='Budget_ID_7', 
    how='left',
    suffixes=('', '_fiscal') 
)

# --- 2. Handle unmatched rows with a second merge on Budget_ID_4 (Project Code level) ---
matched_step1 = merged_step1[merged_step1['monetary_amount'].notna()].copy()

unmatched_bill = merged_step1[merged_step1['monetary_amount'].isna()][bill_df_new.columns].copy()

rescue_merge = pd.merge(
    unmatched_bill,
    fiscal_2023,
    left_on='Budget_ID',
    right_on='Budget_ID_4',
    how='left',
    suffixes=('', '_fiscal')
)

# --- 3. Combine results and clean up ---
final_filtered_df = pd.concat([matched_step1, rescue_merge], ignore_index=True)

final_filtered_df = final_filtered_df.drop_duplicates()

print(f"Complete : {len(final_filtered_df)}")
print(f"Matched : {final_filtered_df['monetary_amount'].notna().sum()} ")

final_filtered_df.head()

Complete : 11525
Matched : 9059 


,bill_project_code,object_code,revised_amount,business_unit,budget_reference,fund_code,join_code,clean_bu,clean_ref,clean_fund,...,year_of_enactment,monetary_amount,clean_bu_fiscal,clean_ref_fiscal,clean_fund_fiscal,prog_str,obj_7,proj_4,Budget_ID_7,Budget_ID_4
0,NaN,1200010.0,3365000,1111,1,264,1200010,1111,001,0264,...,2023.0,225.00,1111,001,0264,1200010000,1200010,1200,2023-1111-001-0264-1200010,2023-1111-001-0264-1200
1,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,3965.55,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
2,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,52500.00,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
3,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,1356.08,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
4,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,5556.60,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130


In [20]:
final_filtered_df.head()

,bill_project_code,object_code,revised_amount,business_unit,budget_reference,fund_code,join_code,clean_bu,clean_ref,clean_fund,...,year_of_enactment,monetary_amount,clean_bu_fiscal,clean_ref_fiscal,clean_fund_fiscal,prog_str,obj_7,proj_4,Budget_ID_7,Budget_ID_4
0,NaN,1200010.0,3365000,1111,1,264,1200010,1111,001,0264,...,2023.0,225.00,1111,001,0264,1200010000,1200010,1200,2023-1111-001-0264-1200010,2023-1111-001-0264-1200
1,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,3965.55,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
2,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,52500.00,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
3,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,1356.08,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130
4,NaN,1130010.0,78520000,1111,1,735,1130010,1111,001,0735,...,2023.0,5556.60,1111,001,0735,1130010000,1130010,1130,2023-1111-001-0735-1130010,2023-1111-001-0735-1130


In [21]:
# Save the merged dataframe to a new CSV file for further analysis
final_filtered_df.to_csv("Merged_Budget_Expenditure_Analysis.csv", index=False, encoding='utf-8-sig')

In [22]:
# 1. Extract necessary columns for the final slim report
cols_to_keep = [
    'Budget_ID',
    'year_of_enactment',  
    'business_unit',      
    'agency_name',        
    'revised_amount',     
    'monetary_amount',    
    'VENDOR_NAME',        
    'accounting_date'     
]

final_slim_df = final_filtered_df[[c for c in cols_to_keep if c in final_filtered_df.columns]].copy()

# 2. Rename columns to match the desired output format
final_slim_df = final_slim_df.rename(columns={
    'year_of_enactment': 'Budget_Enactment_Year',
    'revised_amount': 'Budget_Amount',
    'monetary_amount': 'Actual_Spending',
    'accounting_date': 'Payment_Date'
})

# 3. Sort by year and ID
# Consider the possibility that budgets from years other than 2023 may be mixed in (performance side), and prioritize the budget year (Budget_Enactment_Year)
final_slim_df = final_slim_df.sort_values(['Budget_Enactment_Year', 'Budget_ID'])

# 4. Save as CSV
final_slim_df.to_csv("Strict_Budget_Spending_Report.csv", index=False, encoding='utf-8-sig')

# Check the final output
print(f"Final Report Rows: {len(final_slim_df)}")
print(final_slim_df.head())

Final Report Rows: 11525
                    Budget_ID  Budget_Enactment_Year business_unit  \
6370  2023-1111-001-0069-1125                 2023.0          1111   
6371  2023-1111-001-0069-1125                 2023.0          1111   
6372  2023-1111-001-0069-1125                 2023.0          1111   
6373  2023-1111-001-0069-1125                 2023.0          1111   
6374  2023-1111-001-0069-1125                 2023.0          1111   

                         agency_name  Budget_Amount  Actual_Spending  \
6370  Bus., Consumer Srvcs & Housing       21295000            41.50   
6371  Bus., Consumer Srvcs & Housing       21295000           173.27   
6372  Bus., Consumer Srvcs & Housing       21295000            92.63   
6373  Bus., Consumer Srvcs & Housing       21295000            45.95   
6374  Bus., Consumer Srvcs & Housing       21295000           399.00   

                      VENDOR_NAME Payment_Date  
6370      LAWRENCE TIRE & SERVICE    2023/7/19  
6371       PIERSON AUTO

## 2. Expand all fiscal data

In [23]:
import glob
import os

# 1. Prepare an empty list to collect monthly merged dataframes
final_combined_list = []

# 2. Designate the path to the folder containing the actual expenditure data files (Vendor_FY2*P*.csv)
file_pattern = "Vendor_FY2*P*.csv" 
files = glob.glob(file_pattern)
files.sort()

print(f"Total {len(files)} files to process...")

for file in files:
    print(f"Processing: {file}")
    temp_fiscal = pd.read_csv(file, low_memory=False)

    temp_fiscal = temp_fiscal[temp_fiscal['year_of_enactment'] == 2023].copy()
    
    temp_fiscal['clean_bu'] = temp_fiscal['business_unit'].apply(lambda x: pad_code(x, 4))
    temp_fiscal['clean_ref'] = temp_fiscal['budget_reference'].apply(lambda x: pad_code(x, 3))
    temp_fiscal['clean_fund'] = temp_fiscal['fund_code'].apply(lambda x: pad_code(x, 4))
    
    temp_fiscal['prog_str'] = temp_fiscal['program_code'].astype(str).str.replace('.0', '').str.strip()
    temp_fiscal['Budget_ID_7'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:7]
    temp_fiscal['Budget_ID_4'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:4]


    m1 = pd.merge(bill_df_new, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_7', how='inner', suffixes=('', '_f'))
    matched_ids = m1['Budget_ID'].unique()
    unmatched_bill = bill_df_new[~bill_df_new['Budget_ID'].isin(matched_ids)]
    m2 = pd.merge(unmatched_bill, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_4', how='inner', suffixes=('', '_f'))
    
    final_combined_list.append(m1)
    final_combined_list.append(m2)

# 3. Combine all monthly dataframes into one big dataframe
full_year_df = pd.concat(final_combined_list, ignore_index=True)

# 4. Finally, merge with the original master to ensure all budget items are included, even those without any spending
final_report = pd.merge(bill_df_new, full_year_df, on='Budget_ID', how='left', suffixes=('', '_final'))

Total 30 files to process...
Processing: Vendor_FY23P01.csv
Processing: Vendor_FY23P02.csv
Processing: Vendor_FY23P03.csv
Processing: Vendor_FY23P04.csv
Processing: Vendor_FY23P05.csv
Processing: Vendor_FY23P06.csv
Processing: Vendor_FY23P07.csv
Processing: Vendor_FY23P08.csv
Processing: Vendor_FY23P09.csv
Processing: Vendor_FY23P10.csv
Processing: Vendor_FY23P11.csv
Processing: Vendor_FY23P12.csv
Processing: Vendor_FY24P01.csv
Processing: Vendor_FY24P02.csv
Processing: Vendor_FY24P03.csv
Processing: Vendor_FY24P04.csv
Processing: Vendor_FY24P05.csv
Processing: Vendor_FY24P06.csv
Processing: Vendor_FY24P07.csv
Processing: Vendor_FY24P08.csv
Processing: Vendor_FY24P09.csv
Processing: Vendor_FY24P10.csv
Processing: Vendor_FY24P11.csv
Processing: Vendor_FY24P12.csv
Processing: Vendor_FY25P01.csv
Processing: Vendor_FY25P02.csv
Processing: Vendor_FY25P03.csv
Processing: Vendor_FY25P04.csv
Processing: Vendor_FY25P05.csv
Processing: Vendor_FY25P06.csv


In [24]:
# --- 1. Order the columns ---
keep_cols = [
    'Budget_ID', 'year_of_enactment', 'business_unit', 'agency_name',
    'revised_amount', 'monetary_amount', 'VENDOR_NAME', 'accounting_date'
]
final_report_clean = final_report[[c for c in keep_cols if c in final_report.columns]].copy()

# --- 2. Create summary report ---
summary_report = final_report_clean.groupby(['Budget_ID', 'year_of_enactment']).agg({
    'revised_amount': 'first',      
    'monetary_amount': 'sum',       
    'agency_name': 'first'          
}).reset_index()

# Calculate Spending Rate
summary_report['Spending_Rate'] = (summary_report['monetary_amount'] / summary_report['revised_amount']) * 100

# --- 3. Save

summary_report.to_csv("Budget_Spending_Summary_FY23-25.csv", index=False, encoding='utf-8-sig')

print("Complete! The summary file (Budget_Spending_Summary_FY23-25.csv) has been created.")

Complete! The summary file (Budget_Spending_Summary_FY23-25.csv) has been created.


### Make a Crosswalk table 

In [ ]:
# 1. Extract rows where monetary_amount is not NaN to create a crosswalk table
crosswalk_df = final_report[final_report['monetary_amount'].notna()].copy()

# 2. Extract only the necessary columns for the crosswalk table
crosswalk_table = crosswalk_df[[
    'Budget_ID', 
    'program_code',         
    'program_description'   
]].drop_duplicates()        

# 3. CSVとして保存
crosswalk_table.to_csv("Final_Crosswalk_Table_FY23-25.csv", index=False, encoding='utf-8-sig')

print(f"Crosswalk table has been created with {len(crosswalk_table)} valid pairs.")

Crosswalk table has been created with 1123 valid pairs.
The number of pairs: 1123 
